# 응집도 스케일 검증 실험

격자 크기에 따라 응집 강도(Bind)가 유지되는지 검증합니다.
- 격자: 64×64, 128×128, 256×256, 512×512
- 병렬 처리로 빠른 실행 (4 workers)
- 실시간 진행 상황 로그 표시

## 1. 환경 설정 및 임포트

In [ ]:
import numpy as np
import json
import sys
import time
from multiprocessing import Pool
from datetime import datetime
import os

# Perception_theory 모듈 경로 설정
perception_path = '/Users/ojaehong/Perception/Perception_theory'
if perception_path not in sys.path:
    sys.path.insert(0, perception_path)

from scc.graph import GraphState
from scc.params import ParameterRegistry
from scc.optimizer import find_formation

print(f"✓ 임포트 완료 (경로: {perception_path})")

## 2. 핵심 함수 정의

In [ ]:
def test_cohesion_at_size(grid_size, volume_fraction=0.25):
    """
    Single grid size에서 K=1 형성체의 응집도 측정

    Returns:
        dict: {grid_size, n, bind, sep, inside, persist, u_max, u_min, u_mean, u_std, energy, time_sec, n_iter, converged}
    """
    start_time = time.time()
    timestamp = datetime.now().strftime("%H:%M:%S")

    graph = GraphState.grid_2d(grid_size, grid_size)
    n = graph.n

    print(f"[{timestamp}] {grid_size:>3}x{grid_size:<3} (n={n:>6}) 시작...", flush=True)

    params = ParameterRegistry(volume_fraction=volume_fraction)
    result = find_formation(graph, params, verbose=False)

    u = result.u
    diag = result.diagnostics

    elapsed = time.time() - start_time
    timestamp = datetime.now().strftime("%H:%M:%S")

    print(f"[{timestamp}] {grid_size:>3}x{grid_size:<3} 완료: E={result.energy:>10.6f}, "
          f"Bind={diag.bind:>7.4f}, Sep={diag.sep:>7.4f}, "
          f"Inside={diag.inside:>7.4f}, Persist={diag.persist:>7.4f}, "
          f"시간={elapsed:>6.1f}s, iter={result.n_iter}", flush=True)

    return {
        'grid_size': grid_size,
        'n': n,
        'volume_fraction': volume_fraction,
        'bind': float(diag.bind),
        'sep': float(diag.sep),
        'inside': float(diag.inside),
        'persist': float(diag.persist),
        'u_max': float(np.max(u)),
        'u_min': float(np.min(u)),
        'u_mean': float(np.mean(u)),
        'u_std': float(np.std(u)),
        'energy': float(result.energy),
        'time_sec': elapsed,
        'n_iter': result.n_iter,
        'converged': result.converged
    }


def _worker_wrapper(grid_size, volume_fraction):
    """Wrapper for multiprocessing pool."""
    try:
        return test_cohesion_at_size(grid_size, volume_fraction=volume_fraction)
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print(f"[ERROR] {grid_size}x{grid_size}: {str(e)[:80]}", flush=True)
        return None


print("✓ 함수 정의 완료")

## 3. 실험 파라미터 설정

In [ ]:
# 실험 파라미터
max_size = 256          # 최대 격자 크기 (64, 128, 256, 512)
c_ref = 0.25            # 부피 분수 (응집 강도 기준)
n_workers = 4           # 병렬 처리 워커 수
output_path = 'experiments/results/cohesion_scale_test.json'

# 격자 크기 선택
available_sizes = [64, 128, 256, 512]
grid_sizes = [s for s in available_sizes if s <= max_size]

print(f"실험 설정:")
print(f"  격자 크기: {grid_sizes}")
print(f"  부피 분수: {c_ref}")
print(f"  워커 수: {n_workers}")
print(f"  출력 경로: {output_path}")
print()

## 4. 실험 실행 (병렬 처리)

In [ ]:
print("="*130)
print(f"응집도 스케일 검증 (c_ref={c_ref}, workers={n_workers})")
print(f"시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*130)
print()

results = {}
start_total = time.time()

try:
    # 병렬 처리
    with Pool(processes=n_workers) as pool:
        result_list = pool.starmap(
            _worker_wrapper,
            [(gs, c_ref) for gs in grid_sizes]
        )

    # 결과 수집
    for result in result_list:
        if result is not None:
            key = f'{result["grid_size"]}x{result["grid_size"]}'
            results[key] = result

except KeyboardInterrupt:
    print(f"\n\n중단됨 (by user)")
    print("="*130)

elapsed_total = time.time() - start_total
print(f"\n✓ 실험 완료")
print(f"총 실행 시간: {elapsed_total:.1f}s")
print()

## 5. 결과 저장

In [ ]:
# 출력 디렉토리 생성
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# JSON 저장
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ 저장: {output_path}")
print()

## 6. 분석 및 시각화

In [ ]:
print("="*130)
print("분석")
print("="*130)
print()

if len(results) < 1:
    print("결과 없음.")
else:
    sorted_keys = sorted(results.keys())
    bind_vals = [results[k]['bind'] for k in sorted_keys]
    sep_vals = [results[k]['sep'] for k in sorted_keys]
    time_vals = [results[k]['time_sec'] for k in sorted_keys]

    # 테이블 출력
    print(f"{'Grid':>10} {'Bind':>10} {'Sep':>10} {'Inside':>10} {'Persist':>10} {'Energy':>12} {'Time(s)':>10} {'Iter':>8}")
    print("-"*130)
    for k in sorted_keys:
        r = results[k]
        print(f"{k:>10} {r['bind']:>10.4f} {r['sep']:>10.4f} {r['inside']:>10.4f} {r['persist']:>10.4f} "
              f"{r['energy']:>12.6f} {r['time_sec']:>10.1f} {r['n_iter']:>8d}")

    # 통계
    print(f"\n{'─'*130}")
    print("통계:")
    print(f"  Bind:  min={min(bind_vals):.4f}, max={max(bind_vals):.4f}, "
          f"mean={np.mean(bind_vals):.4f}, std={np.std(bind_vals):.4f}")
    print(f"  Sep:   min={min(sep_vals):.4f}, max={max(sep_vals):.4f}, "
          f"mean={np.mean(sep_vals):.4f}, std={np.std(sep_vals):.4f}")
    print(f"  Time:  total={sum(time_vals):.1f}s, mean={np.mean(time_vals):.1f}s/grid")

    bind_change = (bind_vals[-1] - bind_vals[0]) / bind_vals[0] * 100 if bind_vals[0] != 0 else 0
    sep_change = (sep_vals[-1] - sep_vals[0]) / sep_vals[0] * 100 if sep_vals[0] != 0 else 0

    # 결론
    print(f"\n{'─'*130}")
    print("결론:")

    if min(bind_vals) > 0.5:
        print(f"  ✓ 강한 응집: 모든 격자에서 Bind > 0.5")
    elif min(bind_vals) > 0.3:
        print(f"  △ 중간 응집: 모든 격자에서 0.3 < Bind < 0.5")
    else:
        print(f"  ✗ 약한 응집: Bind < 0.3 (최소값={min(bind_vals):.4f})")

    if abs(bind_change) < 10:
        print(f"  ✓ 안정적: Bind 변화 {bind_change:+.1f}% (< 10%)")
    else:
        print(f"  △ 변동: Bind 변화 {bind_change:+.1f}%")

    if all(sep < 0.5 for sep in sep_vals):
        print(f"  ✓ 분리 안정: 모든 격자에서 Sep < 0.5")
    else:
        print(f"  △ 분리 증가: 일부 격자에서 Sep > 0.5")

    print(f"\n{'='*130}\n")

## 7. 그래프 시각화

In [ ]:
import matplotlib.pyplot as plt

if len(results) >= 2:
    sorted_keys = sorted(results.keys())
    grid_labels = [k.replace('x', '×') for k in sorted_keys]
    
    bind_vals = [results[k]['bind'] for k in sorted_keys]
    sep_vals = [results[k]['sep'] for k in sorted_keys]
    inside_vals = [results[k]['inside'] for k in sorted_keys]
    persist_vals = [results[k]['persist'] for k in sorted_keys]
    energy_vals = [results[k]['energy'] for k in sorted_keys]
    time_vals = [results[k]['time_sec'] for k in sorted_keys]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'응집도 스케일 검증 (c_ref={c_ref})', fontsize=16, fontweight='bold')

    # Bind
    axes[0, 0].plot(grid_labels, bind_vals, 'o-', color='#2E86AB', linewidth=2, markersize=8)
    axes[0, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold=0.5')
    axes[0, 0].set_title('Bind (응집 강도)', fontweight='bold')
    axes[0, 0].set_ylabel('Bind')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    # Sep
    axes[0, 1].plot(grid_labels, sep_vals, 's-', color='#A23B72', linewidth=2, markersize=8)
    axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold=0.5')
    axes[0, 1].set_title('Sep (분리도)', fontweight='bold')
    axes[0, 1].set_ylabel('Sep')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    # Inside
    axes[0, 2].plot(grid_labels, inside_vals, '^-', color='#F18F01', linewidth=2, markersize=8)
    axes[0, 2].set_title('Inside (내부성)', fontweight='bold')
    axes[0, 2].set_ylabel('Inside')
    axes[0, 2].grid(True, alpha=0.3)

    # Persist
    axes[1, 0].plot(grid_labels, persist_vals, 'd-', color='#06A77D', linewidth=2, markersize=8)
    axes[1, 0].set_title('Persist (지속성)', fontweight='bold')
    axes[1, 0].set_ylabel('Persist')
    axes[1, 0].grid(True, alpha=0.3)

    # Energy
    axes[1, 1].plot(grid_labels, energy_vals, 'p-', color='#D62828', linewidth=2, markersize=8)
    axes[1, 1].set_title('Energy (전체 에너지)', fontweight='bold')
    axes[1, 1].set_ylabel('Energy')
    axes[1, 1].grid(True, alpha=0.3)

    # Time
    axes[1, 2].bar(grid_labels, time_vals, color='#1B998B', alpha=0.7, edgecolor='black')
    axes[1, 2].set_title('실행 시간', fontweight='bold')
    axes[1, 2].set_ylabel('Time (s)')
    axes[1, 2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('experiments/results/cohesion_scale_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ 그래프 저장: experiments/results/cohesion_scale_analysis.png")
else:
    print("그래프를 그리기 위해 2개 이상의 결과가 필요합니다.")

## 8. 결과 요약

In [ ]:
print("\n" + "="*130)
print("최종 요약")
print("="*130)
print(f"실험 완료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"총 실행 시간: {elapsed_total:.1f}s")
print(f"결과 저장: {output_path}")
print(f"\n처리된 격자: {', '.join([k for k in sorted(results.keys())])}")
print(f"성공률: {len([r for r in results.values() if r.get('converged')])}/{len(results)}")
print("="*130)